# IAN 601 — Module 12: Cleaning & Exploratory Data Analysis

Real-world data is often incomplete, inconsistent, duplicated, incorrectly typed, or simply surprising.

Data cleaning is the process of finding and addressing those issues deliberately. Exploratory data analysis, or **EDA**, is the process of examining the cleaned data to understand its distributions, categories, unusual values, and possible relationships.

This notebook is self-contained and uses one intentionally messy dataset so the full process can be practiced from audit to validation.

## Learning goals

By the end of this notebook, you should be able to:

1. perform a basic data-quality audit;
2. measure missingness;
3. identify duplicate records;
4. convert numeric and date columns safely;
5. standardize inconsistent text categories;
6. make deliberate missing-value decisions;
7. validate simple ranges and business rules;
8. identify possible outliers with the IQR rule;
9. explore categorical and numeric variables;
10. compare groups;
11. build a compact cleaning pipeline;
12. validate the dataset after cleaning.

## How to use this notebook

Run the cells from top to bottom the first time so that variables and imported libraries are available when later cells need them.

As you work:

1. read the explanation before the code;
2. predict the result;
3. run the cell;
4. inspect the output carefully;
5. change one value, condition, or option;
6. rerun the cell and explain what changed.

For notebooks that load a CSV from GitHub, the dataset is loaded automatically. If you restart the runtime, run the notebook again from the beginning.

## 1. Load the messy dataset

In [ ]:
import pandas as pd
import numpy as np

data_url = "https://raw.githubusercontent.com/ali2449/IAN-601-Introduction-to-Data-Analytics-Methods-and-Approaches/main/IAN601_Module12_Cleaning_EDA_400.csv"
orders = pd.read_csv(data_url)

orders.head()

## 2. Start with a data audit

Do not fix anything yet. First understand what is wrong.

In [ ]:
print("Shape:", orders.shape)
print("\nColumns:", orders.columns.tolist())
print("\nData types:")
print(orders.dtypes)

In [ ]:
orders.head(10)

## 3. Missing values

In [ ]:
missing = pd.DataFrame({
    "missing_count": orders.isna().sum(),
    "missing_percent": (orders.isna().mean() * 100).round(1)
})

missing

## 4. Duplicate records

In [ ]:
print("Exact duplicate rows:", orders.duplicated().sum())
print("Duplicate order IDs:", orders["order_id"].duplicated().sum())

In [ ]:
orders[orders.duplicated(keep=False)].sort_values("order_id")

## 5. Convert data types safely

In [ ]:
clean = orders.copy()

clean["amount"] = pd.to_numeric(
    clean["amount"].astype(str).str.replace("$", "", regex=False),
    errors="coerce"
)

clean["order_date"] = pd.to_datetime(
    clean["order_date"],
    errors="coerce"
)

clean["satisfaction"] = pd.to_numeric(
    clean["satisfaction"],
    errors="coerce"
)

clean.dtypes

## 6. Clean text categories

In [ ]:
clean["region"] = clean["region"].str.strip().str.title()
clean["channel"] = clean["channel"].str.strip().str.lower()
clean["status"] = clean["status"].str.strip().str.lower()

print(clean["region"].value_counts(dropna=False))
print()
print(clean["channel"].value_counts(dropna=False))
print()
print(clean["status"].value_counts(dropna=False))

## 7. Handle missing values deliberately

Here we use simple teaching decisions. In real projects, the decision should depend on context and business meaning.

In [ ]:
clean["city"] = clean["city"].fillna("Unknown")
clean["region"] = clean["region"].fillna("Unknown")
clean["amount"] = clean["amount"].fillna(clean["amount"].median())

clean.isna().sum()

## 8. Validate ranges and business rules

A satisfaction score should be between 1 and 5.

In [ ]:
invalid_satisfaction = clean[
    clean["satisfaction"].notna()
    & ~clean["satisfaction"].between(1, 5)
]

invalid_satisfaction[["order_id", "satisfaction"]]

In [ ]:
clean.loc[
    ~clean["satisfaction"].between(1, 5),
    "satisfaction"
] = np.nan

clean["satisfaction"].describe()

## 9. Outliers are questions, not automatic deletions

In [ ]:
clean["amount"].describe()

### Flag outliers with the IQR rule

In [ ]:
q1 = clean["amount"].quantile(0.25)
q3 = clean["amount"].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outliers = clean[
    (clean["amount"] < lower)
    | (clean["amount"] > upper)
]

print("Lower bound:", round(lower, 2))
print("Upper bound:", round(upper, 2))
print("Flagged rows:", len(outliers))

outliers.head()

## 10. Explore categorical variables

In [ ]:
clean["channel"].value_counts()

In [ ]:
clean["region"].value_counts()

## 11. Explore numeric variables

In [ ]:
clean[["amount", "satisfaction"]].describe()

## 12. Compare groups during EDA

In [ ]:
channel_summary = (
    clean.groupby("channel", as_index=False)
    .agg(
        orders=("order_id", "count"),
        avg_amount=("amount", "mean"),
        avg_satisfaction=("satisfaction", "mean")
    )
)

channel_summary.round(2)

In [ ]:
region_summary = (
    clean.groupby("region", as_index=False)
    .agg(
        orders=("order_id", "count"),
        total_amount=("amount", "sum")
    )
    .sort_values("total_amount", ascending=False)
)

region_summary.round(2)

## 13. Mini cleaning pipeline

A compact pipeline makes the sequence easy to review and reuse.

In [ ]:
pipeline = orders.copy()

pipeline["amount"] = pd.to_numeric(
    pipeline["amount"].astype(str).str.replace("$", "", regex=False),
    errors="coerce"
)
pipeline["order_date"] = pd.to_datetime(
    pipeline["order_date"],
    errors="coerce"
)
pipeline["satisfaction"] = pd.to_numeric(
    pipeline["satisfaction"],
    errors="coerce"
)

pipeline["region"] = pipeline["region"].str.strip().str.title()
pipeline["channel"] = pipeline["channel"].str.strip().str.lower()
pipeline["status"] = pipeline["status"].str.strip().str.lower()

pipeline = pipeline.drop_duplicates().copy()

pipeline["city"] = pipeline["city"].fillna("Unknown")
pipeline["region"] = pipeline["region"].fillna("Unknown")
pipeline["amount"] = pipeline["amount"].fillna(
    pipeline["amount"].median()
)

pipeline.loc[
    ~pipeline["satisfaction"].between(1, 5),
    "satisfaction"
] = np.nan

pipeline.head()

## 14. Check the data after cleaning

In [ ]:
print("Rows:", len(pipeline))
print("Duplicate rows:", pipeline.duplicated().sum())
print("Missing amount:", pipeline["amount"].isna().sum())
print("Invalid dates:", pipeline["order_date"].isna().sum())
print("Invalid satisfaction values:",
      (~pipeline["satisfaction"].dropna().between(1, 5)).sum())

## Practice challenge

Try these on `pipeline`:

1. Find the three channels with the highest total amount.
2. Compare average satisfaction by region.
3. Count orders by status.
4. Find the five largest orders.
5. Decide whether you would remove the amount outliers. Explain why.

In [ ]:
# 1. Total amount by channel


# 2. Average satisfaction by region


# 3. Orders by status


# 4. Five largest orders


# 5. Write your decision about outliers as a comment


## Summary and self-check

Cleaning is not the same as deleting inconvenient rows. Every transformation should have a reason that can be explained.

Check that you can answer:

- What should be inspected before changing the data?
- Why is missingness measured by both count and percentage?
- Why can `errors="coerce"` be useful during conversion?
- Why should text categories be standardized?
- Why is an outlier not automatically an error?
- What does the IQR rule flag?
- Why should the cleaned dataset be validated again at the end?

A trustworthy analysis begins with a trustworthy understanding of the data.